# How long does a Moran process take on a graph of N nodes?

Two questions, and they are not the same one:

- **The biology.** How many steps does an invasion take, and how does that scale with `N`?
- **The wait.** How many *seconds* do I sit there for, and what does a batch at N=2000 or
  N=10000 cost in core-hours?

## Three quantities, three names, never interchange them

The word "time" covers three different things in this notebook, so each one has a fixed name
and nothing is ever called just "time":

| name | what it is | over which runs | units |
|---|---|---|---|
| **fixation steps** | steps until the mutant takes over | only runs that *fixated* | steps |
| **absorption steps** | steps until the run ends either way | *all* runs (fixation or extinction) | steps |
| **simulation time** | wall-clock spent in the C++ absorption loop | *all* runs | seconds |

- **Fixation steps is the biology**: how long a successful invasion takes. It is what
  `graph_statistics.csv` publishes as `mean_steps`, because `io.build_graph_statistics`
  aggregates `steps_success = when(fixation).then(steps)`.
- **Absorption steps is the work done**: every run is paid for, including the ~89% that go
  extinct quickly. At rho ~ 0.11 the two differ by a factor of ~8, and they do **not** share an
  exponent.
- **Simulation time is the wait**, and it is measured, not derived. The `duration` column in the
  raw shards is a `steady_clock` reading around each run's absorption loop
  (`moran_core.cpp:243`), so seconds are a first-class measured quantity here rather than
  steps divided by an assumed throughput.

## Design

**Ladder.** 13 log-spaced sizes, 10 to 1000. Log spacing because the fit consumes lever arm in
`log N`: linear spacing (10, 20, ... 1000) puts 90% of its points in the last half-decade, which
is also where each point is most expensive.

**Topology.** `E = round(1.1 * N)`, i.e. mean degree `k = 2.2` held constant. This matches the real
`avian_r4_l7` graph (N=31, E=34, k=2.19) at *every* rung. Constant *density* would not: density
0.073 means k=2.2 at N=31 but k=73 at N=1000, so a fit through it would mix N with connectivity,
and the zoo would be 1.5 GB instead of ~50 MB.

## What the prior batch does and does not tell you

`2026-06-10_scaling_study_6` covers N=10..100 at constant *density*, and its sparsest cell is a
spanning tree (`E = N-1`, k=2.0) fitting `N^2.281` on fixation steps.

That cell is a **different graph family**, not a baseline to reproduce: k=2.0 versus k=2.2.
Section 10 measures how far apart they actually are, which is a result in its own right. If a 10%
edge surplus moves the answer a lot, this exponent will not transfer to the respiratory graphs
unless their mean degree matches too.

## Notebook layout

Sections 0 to 5 **launch** a batch and are gated off (`BUILD_ZOO`, `SUBMIT`). Everything from
**Analysis** down reads a finished batch and is **standalone**: it re-imports what it needs and
reads its parameters from the batch's own `batch_info.json`, so you can restart the kernel and
run from the Analysis header without touching anything above it.

## Section 0 - Setup

Every import in the notebook, build side and analysis side. Nothing further down imports anything,
so the Analysis half re-enters cleanly after a kernel restart: run this cell, Section 1 and
Section 2, then jump to Analysis.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

# Every import the notebook uses, build side and analysis side, lives here. The
# analysis half is meant to be run on its own after a kernel restart (Setup ->
# Section 1 -> Section 2 -> jump to Analysis), so nothing below may import.
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from moran_process.core.graph_zoo import GraphZoo
from moran_process.core.population_graph import PopulationGraph
from moran_process.pipeline.process_lab import ProcessLab
from moran_process.pipeline.post_batch import post_batch_status
from moran_process.analysis.analysis_utils.io import load_graph_statistics
from moran_process.simulations.cpp_moran_wrapper import CppMoranProcess

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
SIM_DATA = PROJECT_ROOT / "simulation_data"
print(f"Project root: {PROJECT_ROOT}")

## Section 1 - Design constants

Everything that defines the experiment lives here. Nothing below this cell hard-codes a size,
a repeat count, or a job count.

In [ ]:
DATE = "2026_08_26"
DESCRIPTION = (
    "Scaling study: absorption time vs population size N, at constant mean degree "
    "k=2.2 (avian-like sparsity). Log-spaced ladder N=10..1000, r=1.1."
)
NOTES = (
    "Companion to 2026-06-10_scaling_study_6, which covers N=10..100 at constant "
    "density. NOT a replication: June's sparsest cell is a spanning tree (E=N-1, "
    "k~2.0); this zoo uses E=round(1.1N) (k=2.2). A pilot shows that 10% edge "
    "surplus cuts absorption time ~17x at N=100, so the two are different regimes."
)

# --- the ladder, log-spaced over two decades -------------------------------------
LADDER = [10, 15, 22, 32, 46, 68, 100, 150, 220, 320, 460, 680, 1000]

# One batch. A pilot (see Section 2) puts the whole study at ~1.3 core-hours with a
# worst single task under 4 min, so the cost-tiered split this notebook originally
# carried was solving a problem that does not exist at k=2.2. Tiering matters when
# per-simulation cost spans orders of magnitude across the ladder, because
# _create_task_list balances workers by SIM COUNT, not by cost. Here the spread is
# ~1100x but the absolute worst case is minutes, so it is not worth three batches.
# If you raise n_repeats or extend the ladder far past 1000, re-check that.
N_JOBS = 1000
MEMORY = "4GB"

# --- topology: constant mean degree, avian-like ----------------------------------
K_MEAN = 2.2  # avian_r4_l7 has N=31, E=34 -> k=2.194


def n_edges_for(n_nodes: int) -> int:
    """Edges at constant mean degree k, floored at a spanning tree."""
    return max(n_nodes - 1, round(K_MEAN * n_nodes / 2))


N_SEEDS = 200  # random graphs per rung
BASELINES = ("cycle", "star", "complete")  # one of each per rung

# --- simulation ------------------------------------------------------------------
R_VALUES = [1.1]
N_REPEATS = 10_000
SEED = 42  # batch_seed: simulation RNG
GRAPH_ZOO_SEED = 42  # random-graph topology RNG
ENGINE = "cpp"
QUEUE = "gsla-cpu"

# The engine default is 1e6. The pilot puts the largest MEAN at ~1.1e5 (N=1000), so
# the default would not censor the mean -- but absorption time is heavy-tailed and
# individual fixating runs run far above it, so some would clip. A censored run is
# written as (fixation=False, steps=max_steps), indistinguishable from extinction,
# and it biases mean_steps DOWN at exactly the large-N end being extrapolated from.
# 1e9 costs nothing here and removes the failure mode. Section 7 asserts nothing hit it.
MAX_STEPS = 1_000_000_000

# --- reference batch -------------------------------------------------------------
JUNE_BATCH = SIM_DATA / "2026-06-10_scaling_study_6"

BATCH_NAME = f"{DATE}-size-study"
BATCH_DIR = SIM_DATA / BATCH_NAME

print(f"Ladder ({len(LADDER)} rungs): {LADDER}")
print(f"{'N':>6} {'E':>7} {'k':>6}")
for n in LADDER:
    e = n_edges_for(n)
    print(f"{n:>6} {e:>7} {2 * e / n:>6.2f}")
print(f"\nBatch: {BATCH_NAME}")
print(f"Graphs: {len(LADDER) * (N_SEEDS + len(BASELINES)):,}")

## Section 2 - Cost preview, before anything is built or submitted

The scaling study is itself subject to the scaling law, so price it first.

The numbers below are a **pilot measured directly at k=2.2** (5 graphs per rung, 400 repeats,
r=1.1) rather than extrapolated from June, because the June tree cell overestimates cost here by
~240x. They are a budgeting prior only: Section 9 re-fits the exponent from the real batch and
Section 11 re-measures throughput.

In [ ]:
# Pilot measured on a login node, 2026-08-23: 5 random graphs per rung at k=2.2,
# 400 repeats each, r=1.1, cpp engine. "absorption_steps" = mean steps over ALL
# runs (see the terminology table at the top); it is the work-done quantity, and
# the only one of the three a budget can be built from.
PILOT = pd.DataFrame(
    {
        "N": [10, 15, 22, 32, 46, 68, 100, 150, 220, 320, 460, 680, 1000],
        "absorption_steps": [98, 242, 464, 936, 1491, 3734, 5674, 9812, 17188, 27989,
                             42427, 77618, 111222],
        "M_steps_per_sec": [77.0, 87.7, 93.7, 99.2, 103.0, 107.9, 110.8, 115.0,
                            120.5, 121.7, 123.9, 126.7, 129.5],
    }
).set_index("N")

_x, _y = np.log(PILOT.index.values.astype(float)), np.log(PILOT["absorption_steps"].values)
PRIOR_ALPHA, _ic = np.polyfit(_x, _y, 1)
PRIOR_C = np.exp(_ic)
STEPS_PER_SEC = PILOT["M_steps_per_sec"].iloc[-1] * 1e6  # large-N rate; cost lives there

print(f"pilot prior: T = {PRIOR_C:.3f} * N^{PRIOR_ALPHA:.3f}")
print(f"pilot throughput: {PILOT.M_steps_per_sec.iloc[0]:.0f} M/s at N=10 "
      f"-> {PILOT.M_steps_per_sec.iloc[-1]:.0f} M/s at N=1000 (RISES with N)")
print(
    "\nNote: this does NOT match June's spanning-tree cell (alpha=2.281, T=97,560 at\n"
    "N=100). At k=2.2 the pilot gives 5,674 at N=100 -- 17x faster. A 10% edge surplus\n"
    "over a tree destroys the suppression, so the tree cell is not a proxy for avian.\n"
)


def prior_steps(n):
    return PRIOR_C * np.asarray(n, dtype=float) ** PRIOR_ALPHA


n_graphs_per_rung = N_SEEDS + len(BASELINES)
preview = pd.DataFrame(
    {
        "N": LADDER,
        "E": [n_edges_for(n) for n in LADDER],
        "graphs": n_graphs_per_rung,
        "absorption_steps": [PILOT["absorption_steps"].get(n, prior_steps(n))
                             for n in LADDER],
    }
)
preview["sec_per_graph"] = (
    preview.absorption_steps * N_REPEATS * len(R_VALUES) / STEPS_PER_SEC
)
preview["core_hours"] = preview.sec_per_graph * preview.graphs / 3600

with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
    print(preview.to_string(index=False))

n_sims = preview.graphs.sum() * N_REPEATS * len(R_VALUES)
# _create_task_list gives every worker an equal SIM COUNT. When that share is below
# N_REPEATS, expensive configs get split across workers; above it, one worker eats a
# whole config. Either way what matters is the wall time of the unluckiest worker.
share = n_sims / N_JOBS
worst = share * preview.absorption_steps.max() / STEPS_PER_SEC
print(f"\nTOTAL: {preview.core_hours.sum():.2f} core-hours | {n_sims:,} simulations")
print(f"{N_JOBS} jobs | {share:,.0f} sims/worker | worst worker ~{worst / 60:.1f} min")
print(f"Longest single config (N={LADDER[-1]}): "
      f"{preview.absorption_steps.max() * N_REPEATS / STEPS_PER_SEC:.0f} s")

over = preview[preview.absorption_steps > 1e6].N.tolist()
print(f"\nRungs whose MEAN exceeds the 1e6 engine default: {over or 'none'}")
print(f"MAX_STEPS is set to {MAX_STEPS:,} (tail insurance, see Section 7)")

## Section 3 - Build the zoo

Random graphs at `E = round(1.1N)` plus one cycle, star and complete per rung.

Cycle and star are both k~2 and bracket the sparse regime: the cycle is the slow, neutral-ish
case, the star the extreme amplifier. Complete is the dense reference with known theory. They
give the topology-dependence of alpha for free, since they cost almost nothing to add.

In [ ]:
# The batch below is already built, submitted and aggregated, so on an analysis pass
# this cell is pure waste: 2,639 graphs, and the complete-graph baselines alone carry
# ~500k edges at N=1000. Flip to True only when launching a NEW batch (change DATE
# first, or you will overwrite the existing zoo.pkl in Section 5).
BUILD_ZOO = False

if not BUILD_ZOO:
    print("BUILD_ZOO is False - Sections 3-5 skipped. Go straight to Analysis.")
else:
    zoo = GraphZoo(name=BATCH_NAME)
    for n in LADDER:
        if "cycle" in BASELINES:
            zoo.add(PopulationGraph.cycle_graph(n_nodes=n))
        if "star" in BASELINES:
            zoo.add(PopulationGraph.star_graph(n_nodes=n))
        if "complete" in BASELINES:
            zoo.add(PopulationGraph.complete_graph(n_nodes=n))
        e = n_edges_for(n)
        for seed in range(N_SEEDS):
            zoo.add(
                PopulationGraph.random_connected_graph(
                    n_nodes=n, n_edges=e, seed=GRAPH_ZOO_SEED * 100_003 + seed
                )
            )

    total_edges = sum(g.graph.number_of_edges() for g in zoo)
    print(f"{len(zoo):,} graphs, {total_edges:,} edges")
    # June's zoo.pkl came out at 54 bytes/edge; use that to predict pickle size.
    print(f"~{total_edges * 54 / 1e6:.0f} MB pickled")
    print(f"\nNote: the complete-graph baselines carry most of those edges "
          f"({sum(g.graph.number_of_edges() for g in zoo if g.category == 'Complete'):,}); "
          f"the k=2.2 study graphs are only "
          f"{sum(g.graph.number_of_edges() for g in zoo if g.category == 'Random'):,}.")

## Section 4 - Inspect before committing

Confirm the mean degree really is constant across the ladder. If `k` drifts, the fitted exponent
is measuring connectivity as well as N and the whole study is confounded.

In [ ]:
if not BUILD_ZOO:
    print("BUILD_ZOO is False - nothing to inspect.")
else:
    zoo_summary = pd.DataFrame(
        [
            dict(
                category=g.category,
                n_nodes=g.graph.number_of_nodes(),
                n_edges=g.graph.number_of_edges(),
                k=2 * g.graph.number_of_edges() / g.graph.number_of_nodes(),
            )
            for g in zoo
        ]
    )

    rnd = zoo_summary[zoo_summary.category == "Random"]
    print("Random graphs, mean degree by rung (must be flat at k=2.2):")
    print(rnd.groupby("n_nodes")["k"].agg(["mean", "min", "max", "count"]).round(3).to_string())

    # If k drifts across the ladder, the fitted exponent measures connectivity as well as
    # N and the study is confounded. Tolerance is loose at small N only because rounding
    # E to an integer cannot hit 2.2 exactly there (N=15 -> E=16 -> k=2.13).
    assert np.allclose(rnd.groupby("n_nodes")["k"].mean(), K_MEAN, atol=0.12), (
        "mean degree is not constant across the ladder"
    )
    print("\nGraphs by category:")
    print(zoo_summary.groupby("category").size().to_string())

## Section 5 - Save and submit

**Requires `max_steps` to be plumbed through `submit_jobs` -> `task_manifest` -> `worker_lsf`.**
The engine constructor defaults to 1e6 and neither the pipeline nor the worker currently passes
anything else. At k=2.2 that default would not censor any rung's *mean* (largest is ~1.1e5 at
N=1000), but absorption time is heavy-tailed and individual fixating runs go far above the mean,
so some would clip. Section 7 checks.

Submission is in a separate cell from the save, and guarded, so a stray Run All does not fire a
job array.

In [ ]:
ZOO_PATH = BATCH_DIR / "zoo.pkl"

# Defined unconditionally (the submit cell needs the path), written only when the
# zoo was actually rebuilt -- otherwise this would clobber the pickle the finished
# batch was run from.
if not BUILD_ZOO:
    print(f"BUILD_ZOO is False - not writing. Existing: {ZOO_PATH} "
          f"({'present' if ZOO_PATH.exists() else 'MISSING'})")
else:
    BATCH_DIR.mkdir(parents=True, exist_ok=True)
    zoo.save(str(ZOO_PATH))
    print(f"{len(zoo):,} graphs -> {ZOO_PATH}  ({ZOO_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
SUBMIT = False  # the batch below is DONE; flip to True only to launch a new one

if not SUBMIT:
    print("SUBMIT is False - nothing submitted. Set SUBMIT = True to launch.")
else:
    zoo_on_disk = GraphZoo.load(str(ZOO_PATH))
    categories = sorted({g.category for g in zoo_on_disk})
    lab = ProcessLab()
    job_ids = lab.submit_jobs(
        zoo_path=str(ZOO_PATH),
        n_graphs=len(zoo_on_disk),
        r_values=R_VALUES,
        batch_name=BATCH_NAME,
        batch_dir=str(BATCH_DIR),
        n_repeats=N_REPEATS,
        n_requested_jobs=N_JOBS,
        queue=QUEUE,
        memory=MEMORY,
        graph_types=categories,
        node_sizes=LADDER,
        description=DESCRIPTION,
        notes=NOTES,
        batch_seed=SEED,
        engine=ENGINE,
        max_steps=MAX_STEPS,  # <-- needs the plumbing change described above
        zoo_config=dict(
            graph_zoo_seed=GRAPH_ZOO_SEED,
            k_mean=K_MEAN,
            n_seeds=N_SEEDS,
            baselines=list(BASELINES),
            sizes=LADDER,
            edge_rule="E = max(N-1, round(K_MEAN*N/2))",
        ),
    )
    print(job_ids)

---
# Analysis

**This section is standalone.** The bootstrap cell immediately below re-imports everything it
needs and reads the batch's parameters from `batch_info.json`, so after a kernel restart you can
click into it and Run-Below without executing a single cell from the launch half. Nothing here
depends on `DATE`, `LADDER`, `PILOT` or any other notebook global defined above.

That is deliberate: the analysis targets **one specific finished batch**, named explicitly in the
bootstrap. If it inherited `BATCH_NAME` from Section 1, then editing `DATE` to launch a *new*
study would silently repoint every figure below at a directory that does not exist yet.

The parameters come from the batch rather than from the notebook for the same reason. `r`,
`n_repeats`, `max_steps` and the size ladder are properties of **the data that was produced**, not
of whatever the constants cell happens to say today.

Cost: Section 7 makes one lazy polars pass over the raw shards (~26M rows, a grouped aggregation,
a few seconds). Everything after that is file reads and cheap fitting. No simulation, no
re-aggregation in the kernel.

In [ ]:
# =========================================================================
# ANALYSIS BOOTSTRAP - run this first; nothing above it is required.
# =========================================================================
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from moran_process.core.population_graph import PopulationGraph
from moran_process.pipeline.post_batch import post_batch_status
from moran_process.analysis.analysis_utils.io import load_graph_statistics
from moran_process.simulations.cpp_moran_wrapper import CppMoranProcess

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
SIM_DATA = PROJECT_ROOT / "simulation_data"

# The batch under analysis, named explicitly rather than inherited from Section 1,
# so that editing DATE up there to launch a new study cannot silently repoint the
# figures below at a directory that has no results in it yet.
ANALYSIS_BATCH = "2026_08_26-size-study"
BATCH_DIR = SIM_DATA / ANALYSIS_BATCH
JUNE_BATCH = SIM_DATA / "2026-06-10_scaling_study_6"  # the k=2.0 comparison, Section 10

# Parameters come from the BATCH, not from the notebook: r, n_repeats, max_steps
# and the ladder are properties of the data that exists on disk, and batch_info.json
# is what submit_jobs actually recorded when it ran.
INFO = json.loads((BATCH_DIR / "batch_info.json").read_text())
R_VALUES = INFO["simulation"]["r_values"]
N_REPEATS = INFO["simulation"]["n_repeats"]
MAX_STEPS = INFO["simulation"]["max_steps"]
ENGINE = INFO["simulation"]["engine"]
LADDER = INFO["zoo"]["sizes"]
K_MEAN = INFO["zoo"]["k_mean"]
N_SEEDS = INFO["zoo"]["n_seeds"]
BASELINES = tuple(INFO["zoo"]["baselines"])
GRAPHS_PER_RUNG = N_SEEDS + len(BASELINES)


def n_edges_for(n_nodes: int) -> int:
    """Edges at constant mean degree k, floored at a spanning tree."""
    return max(n_nodes - 1, round(K_MEAN * n_nodes / 2))


# The three quantities, named once here and used verbatim everywhere below.
QUANTITIES = {
    "fixation_steps": "steps to fixation, over fixating runs only (the biology)",
    "absorption_steps": "steps to absorption, over all runs (the work done)",
    "sim_seconds": "wall-clock seconds in the C++ loop, over all runs (the wait)",
}

print(f"batch      : {ANALYSIS_BATCH}")
print(f"created    : {INFO['created_at']}   engine: {ENGINE}")
print(f"r          : {R_VALUES}    n_repeats: {N_REPEATS:,}    max_steps: {MAX_STEPS:,}")
print(f"ladder     : {LADDER}")
print(f"per rung   : {GRAPHS_PER_RUNG} graphs ({N_SEEDS} random at k={K_MEAN} + {list(BASELINES)})")
print(f"planned    : {INFO['simulation']['total_simulations']:,} simulations")
print("\nthe three quantities:")
for k, v in QUANTITIES.items():
    print(f"  {k:18s} {v}")

## Section 6 - Batch status and completeness

Two different questions, and `post_batch_status` only answers the first.

- **Are the derived artefacts built?** That is the `steps` dict: aggregate, verify, job speed,
  violin cache.
- **Is the batch actually complete?** That is `report/verification.json`, and on this batch it
  says **FAIL**. Read it before trusting any number below, because `aggregate` does not fail on a
  missing shard, it rolls up whatever exists and produces a clean-looking `graph_statistics.csv`
  either way.

The cell below prints the verdict and then quantifies it, rather than leaving a bare FAIL to be
ignored or over-reacted to.

In [ ]:
status = post_batch_status(str(BATCH_DIR), r_values=R_VALUES)
print(f"kind={status['kind']}  ready={status['ready']}  verify={status['verify_overall']}")
for step, state in status["steps"].items():
    print(f"  {step:14s} {state}")

# post_batch_status reports the VERDICT; the detail lives in the report itself.
report = json.loads((BATCH_DIR / "report" / "verification.json").read_text())
print(f"\nverification.json -> {report['overall']}")
for chk in report["checks"]:
    mark = " " if chk["status"] == "OK" else ">"
    print(f" {mark}[{chk['status']:4s}] {chk['name']:22s} {chk['message']}")

## Section 7 - The one shard pass: all three quantities, and the censoring check

**Run this before looking at any fit.** One lazy pass over the raw shards, producing every
quantity the rest of the notebook uses, per graph.

### Why all three come from here

`graph_statistics.csv` publishes only **fixation steps** (`io.build_graph_statistics` aggregates
`steps_success = when(fixation).then(steps)`). The other two exist only in the raw shards:

| quantity | expression over the shards |
|---|---|
| `fixation_steps` | `mean(steps)` where `fixation` |
| `absorption_steps` | `mean(steps)` over every row |
| `sim_seconds` | `mean(duration)` over every row |

`duration` is a `steady_clock` interval wrapped around each run's absorption loop
(`moran_core.cpp:243`). Two things it does **not** include, both of which matter when you turn it
into a wall-clock promise in Section 12: the O(N) `initialize_random_mutant` before the loop, and
every layer of Python, I/O and job startup outside the engine. Section 12 measures those
separately instead of assuming them away.

### Censoring

`worker_lsf` writes an explicit `censored` column (True when a run was stopped by `max_steps`
rather than absorbing). Without it a truncated run is written as `(fixation=False,
steps=max_steps)`, indistinguishable from a fast extinction, which drags every mean *down* at
exactly the large-N end you extrapolate from. The assert below is the guard.

In [ ]:
shards = BATCH_DIR / "tmp" / "results" / "*.parquet"
lf = pl.scan_parquet(str(shards))
names = lf.collect_schema().names()

# `censored` is written by worker_lsf; batches from before that change lack it, in
# which case fall back to the (exact, same-semantics) steps == max_steps test.
censored_expr = (
    pl.col("censored") if "censored" in names else (pl.col("steps") >= MAX_STEPS)
)

raw = (
    lf.group_by("wl_hash")
    .agg(
        pl.len().alias("n_runs"),
        censored_expr.sum().alias("n_censored"),
        pl.col("steps").max().alias("max_steps_seen"),
        # THE BIOLOGY: steps to fixation, over fixating runs only.
        pl.when(pl.col("fixation")).then(pl.col("steps")).mean().alias("fixation_steps"),
        # THE WORK: steps to absorption, over every run, fixating or not.
        pl.col("steps").mean().alias("absorption_steps"),
        # THE WAIT: measured wall-clock seconds per run, not steps / assumed rate.
        pl.col("duration").mean().alias("sim_seconds"),
        pl.col("duration").sum().alias("sim_seconds_total"),
        pl.col("fixation").mean().alias("rho"),
    )
    .collect()
    .to_pandas()
)
print(f"{'(explicit censored column)' if 'censored' in names else '(derived from steps)'}")
print(f"Runs at the ceiling: {raw.n_censored.sum():,} / {raw.n_runs.sum():,}")
print(f"Largest absorption seen: {raw.max_steps_seen.max():,} steps of a {MAX_STEPS:,} cap")

assert raw.n_censored.sum() == 0, (
    "CENSORED RUNS PRESENT - every quantity is biased low at the affected graphs. "
    "Raise MAX_STEPS and re-run before fitting."
)
print("No censoring. All three quantities below are unbiased.\n")

# Attach graph size. graph_props.csv is the join partner for wl_hash.
props = pd.read_csv(BATCH_DIR / "graph_props.csv")
raw = raw.merge(props[["wl_hash", "category", "n_nodes", "n_edges"]], on="wl_hash")

# Section 6 flagged 32 short cells. They enter the per-rung mean with the same weight
# as a complete one, so quantify the damage rather than assuming either way.
short = raw[raw.n_runs < N_REPEATS]
print(f"Cells short of {N_REPEATS:,} runs: {len(short)} of {len(raw)} "
      f"(smallest {short.n_runs.min():,} runs)" if len(short) else "All cells complete.")
if len(short):
    rnd = raw[raw.category == "Random"]
    full = rnd[rnd.n_runs == N_REPEATS]
    for col in ("fixation_steps", "absorption_steps", "sim_seconds"):
        a = rnd.groupby("n_nodes")[col].mean()
        b = full.groupby("n_nodes")[col].mean()
        print(f"  dropping them shifts {col:18s} by at most "
              f"{((a - b).abs() / b * 100).max():.2f}%")
    print("  -> kept. The short cells are noise, not bias; excluding them changes no "
          "conclusion,\n     and dropping data that does not matter is its own kind of lie.")

# Steps and seconds do not share a sensible float format, so give each its own.
FMT = {
    "fixation_steps": lambda v: f"{v:,.0f}",
    "absorption_steps": lambda v: f"{v:,.0f}",
    "ms_per_run": lambda v: f"{v:,.4f}",
    "sec_per_graph": lambda v: f"{v:,.2f}",
    "rho": lambda v: f"{v:.3f}",
    "fix/abs": lambda v: f"{v:.2f}",
    "M_steps_per_sec": lambda v: f"{v:.1f}",
}

print("\nThe three quantities, random graphs only:")
chk = raw[raw.category == "Random"].groupby("n_nodes")[
    ["fixation_steps", "absorption_steps", "sim_seconds", "rho"]
].mean()
chk["ms_per_run"] = chk.pop("sim_seconds") * 1e3
chk["fix/abs"] = chk.fixation_steps / chk.absorption_steps
chk["M_steps_per_sec"] = chk.absorption_steps / (chk.ms_per_run / 1e3) / 1e6
print(chk.to_string(formatters=FMT))

## Section 8 - Cross-check against the published statistics, and build the curves

`graph_statistics.csv` is the documented reader path, but it carries only one of the three
quantities and it is the conditional one. So this section uses it as a **cross-check** that the
shard pass reproduces `mean_steps` exactly, and then builds the three per-rung curves from the
shard pass.

The cycle / star / complete baselines are simulated alongside but are kept out of the main fit.
June shows the exponent ranging 1.83 (complete) to 2.70 (cycle, star), so it is a property of the
topology family, and pooling them would average distinct laws into a meaningless middle.

In [ ]:
# The published mean_steps is CONDITIONAL on fixation, so it is the partner of
# fixation_steps and of nothing else. If these two disagree, the shard pass is wrong.
stats = load_graph_statistics(str(BATCH_DIR), r_filter=R_VALUES)
print(f"{len(stats):,} (graph, r) rows")
print(stats.groupby("category").size().to_string())

xchk = stats[["wl_hash", "mean_steps"]].merge(raw[["wl_hash", "fixation_steps"]], on="wl_hash")
rel = ((xchk.mean_steps - xchk.fixation_steps).abs() / xchk.fixation_steps).max()
print(f"\ngraph_statistics.mean_steps vs shard-pass fixation_steps: max rel. diff {rel:.2e}")
assert rel < 1e-6, "graph_statistics.mean_steps disagrees with the shard pass"

sparse = raw[raw.category == "Random"]


def series(col):
    """Per-rung mean of `col` across the random graphs, with its standard error."""
    g = sparse.groupby("n_nodes")[col].agg(["mean", "std", "count"])
    g.columns = ["T", "sd", "n_graphs"]
    g["sem"] = g.sd / np.sqrt(g.n_graphs)
    return g


curve_fixation = series("fixation_steps")     # the biology
curve_absorption = series("absorption_steps")  # the work
curve_seconds = series("sim_seconds")          # the wait

summary = pd.DataFrame(
    {
        "fixation_steps": curve_fixation["T"],
        "absorption_steps": curve_absorption["T"],
        "sim_seconds": curve_seconds["T"],
        "graphs": curve_absorption["n_graphs"],
    }
)
summary["ms_per_run"] = summary.pop("sim_seconds") * 1e3
summary["sec_per_graph"] = summary.ms_per_run / 1e3 * N_REPEATS
print(f"\nPer-rung means over the random k=2.2 graphs "
      f"(sec_per_graph = one graph at {N_REPEATS:,} repeats, one core):")
print(summary.to_string(formatters={**FMT, "graphs": lambda v: f"{v:.0f}"}))

# Historical: the Section 2 pilot measured absorption steps on a login node. Guarded,
# because the analysis half must not require the launch half to have been run.
if "PILOT" in globals():
    print("\nMeasured absorption_steps vs the Section 2 pilot prior:")
    for n in curve_absorption.index:
        if n in PILOT.index:
            m, p = curve_absorption.loc[n, "T"], PILOT.loc[n, "absorption_steps"]
            print(f"  N={n:>5}: measured {m:>10,.0f}  pilot {p:>10,.0f}  ratio {m / p:.2f}")
else:
    print("\n(Section 2 not run, so the pilot comparison is skipped - it is historical only.)")

## Section 9 - Linear, polynomial, or exponential?

Three nested claims, discriminated on the same data, for each of the three quantities:

| model | linearised form | signature |
|---|---|---|
| linear | `T ~ N` | power law with `alpha = 1` |
| polynomial | `log T ~ alpha * log N` | straight on log-log |
| exponential | `log T ~ beta * N` | straight on lin-log |

R^2 alone is a weak discriminator over two decades, so this also reports AIC (same response
`log T`, same parameter count, so AIC is directly comparable) and a **curvature test**: fit the
exponent on the bottom half and the top half of the ladder separately.

The curvature test is the one that matters for Section 12. A true power law gives the same
`alpha` on both halves; a drifting `alpha` means the global fit is an average of two different
local behaviours, and extrapolating past N=1000 with it is unjustified however good the global
R^2 looks.

In [ ]:
def fit_line(x, y):
    """OLS slope/intercept plus R^2, AIC and the slope's standard error."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(x)
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    rss = float(resid @ resid)
    r2 = 1 - rss / float(((y - y.mean()) ** 2).sum())
    aic = n * np.log(rss / n) + 2 * 3  # slope, intercept, sigma
    se = np.sqrt(rss / (n - 2) / ((x - x.mean()) ** 2).sum())
    return dict(slope=slope, intercept=intercept, r2=r2, aic=aic, se=se, n=n)


def analyse(curve, label, unit):
    """Discriminate power law vs exponential, and test whether alpha is stable."""
    N = curve.index.values.astype(float)
    logT = np.log(curve["T"].values)
    power, expo = fit_line(np.log(N), logT), fit_line(N, logT)

    a, se = power["slope"], power["se"]
    lo, hi = a - 1.96 * se, a + 1.96 * se
    d_aic = expo["aic"] - power["aic"]

    print(f"=== {label}  [{unit}] ===")
    print(f"  power law    ~ N^{a:.3f} +/- {1.96 * se:.3f}"
          f"   R2={power['r2']:.5f}  AIC={power['aic']:.1f}")
    print(f"  exponential  ~ exp({expo['slope']:.5f} N)"
          f"       R2={expo['r2']:.5f}  AIC={expo['aic']:.1f}")
    print(f"  delta AIC = {abs(d_aic):.1f} favouring "
          f"{'POWER LAW' if d_aic > 0 else 'EXPONENTIAL'}")
    for name, val in [("linear", 1), ("quadratic", 2), ("cubic", 3)]:
        print(f"    {name} (alpha={val})? "
              f"{'consistent' if lo <= val <= hi else 'ruled out'}")

    # Curvature. The upper-half fit is kept because Section 12 extrapolates with it:
    # if alpha drifts, the LOCAL exponent at the top of the ladder is the honest
    # basis for going beyond it, not the global average.
    h = len(N) // 2
    f_lo = fit_line(np.log(N[: h + 1]), logT[: h + 1])
    f_hi = fit_line(np.log(N[h:]), logT[h:])
    drift = abs(f_hi["slope"] - f_lo["slope"])
    pooled = np.hypot(f_lo["se"], f_hi["se"])
    stable = drift < 2 * pooled
    print(f"  curvature: alpha(N<={N[h]:.0f})={f_lo['slope']:.3f}"
          f"  alpha(N>={N[h]:.0f})={f_hi['slope']:.3f}"
          f"  drift={drift:.3f} ({drift / pooled:.1f} sigma)")
    print("  -> " + ("stable; one exponent extrapolates"
                     if stable else "DRIFTING; the global exponent does NOT extrapolate"))
    print()
    return dict(power=power, expo=expo, alpha=a, lo=lo, hi=hi, N=N, logT=logT,
                curve=curve, label=label, unit=unit,
                alpha_hi=f_hi["slope"], alpha_lo=f_lo["slope"], stable=stable,
                split=N[h])


fit_fixation = analyse(curve_fixation, "FIXATION STEPS  (the biology)", "steps")
fit_absorption = analyse(curve_absorption, "ABSORPTION STEPS  (the work)", "steps")
fit_seconds = analyse(curve_seconds, "SIMULATION TIME  (the wait)", "seconds")

FITS = [fit_fixation, fit_absorption, fit_seconds]

print("The three exponents:")
for f in FITS:
    print(f"  {f['label'].split('  ')[0]:20s} alpha = {f['alpha']:.3f} "
          f"+/- {1.96 * f['power']['se']:.3f}")
gap = abs(fit_fixation["alpha"] - fit_absorption["alpha"])
pooled = 1.96 * np.hypot(fit_fixation["power"]["se"], fit_absorption["power"]["se"])
print("\nFixation and absorption steps scale "
      + ("DIFFERENTLY" if gap > pooled else "the same")
      + f" (gap {gap:.3f}, 95% pooled {pooled:.3f}).")
print("Simulation time tracks absorption steps, not fixation steps, because every run "
      "is paid for.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
STEPS_SERIES = [
    (fit_fixation, "fixation steps", "darkorange"),
    (fit_absorption, "absorption steps", "steelblue"),
]
SEC_SERIES = [(fit_seconds, "simulation time", "seagreen")]

# (a) log-log, steps. A power law is a straight line here.
ax = axes[0, 0]
for f, label, c in STEPS_SERIES:
    ax.errorbar(f["N"], f["curve"]["T"], yerr=f["curve"]["sem"], fmt="o", color=c,
                label=f"{label}: $N^{{{f['alpha']:.2f}}}$")
    grid = np.logspace(np.log10(f["N"].min()), np.log10(f["N"].max()), 100)
    ax.plot(grid, np.exp(f["power"]["intercept"]) * grid ** f["alpha"], "-", color=c, lw=1)
ax.set(xscale="log", yscale="log", xlabel="N (nodes)", ylabel="steps",
       title="(a) steps, log-log: straight = polynomial")
ax.legend(fontsize=8)

# (b) log-log, seconds. Its own panel because seconds and steps do not share an axis.
ax = axes[0, 1]
for f, label, c in SEC_SERIES:
    ax.errorbar(f["N"], f["curve"]["T"] * 1e3, yerr=f["curve"]["sem"] * 1e3, fmt="o",
                color=c, label=f"{label}: $N^{{{f['alpha']:.2f}}}$")
    grid = np.logspace(np.log10(f["N"].min()), np.log10(f["N"].max()), 100)
    ax.plot(grid, np.exp(f["power"]["intercept"]) * grid ** f["alpha"] * 1e3, "-",
            color=c, lw=1)
ax.set(xscale="log", yscale="log", xlabel="N (nodes)",
       ylabel="milliseconds per run (1 core)",
       title="(b) THE WAIT: measured wall-clock per run")
ax.legend(fontsize=8)

# (c) lin-log. An exponential is a straight line here; these are visibly curved.
ax = axes[1, 0]
for f, label, c in STEPS_SERIES + SEC_SERIES:
    y = f["curve"]["T"] / f["curve"]["T"].iloc[0]  # normalised: mixed units on one axis
    ax.plot(f["N"], y, "o", color=c, label=label)
    ax.plot(f["N"], np.exp(f["expo"]["intercept"] + f["expo"]["slope"] * f["N"])
            / np.exp(f["expo"]["intercept"] + f["expo"]["slope"] * f["N"][0]),
            "-", color=c, lw=1)
ax.set(yscale="log", xlabel="N (nodes)", ylabel="growth relative to N=10",
       title="(c) lin-log: straight = exponential (it is not)")
ax.legend(fontsize=8)

# (d) local exponent. Flat means a genuine single-exponent power law; this is the
# panel that decides whether Section 12 may extrapolate with the global alpha.
ax = axes[1, 1]
for f, label, c in STEPS_SERIES + SEC_SERIES:
    local = np.diff(f["logT"]) / np.diff(np.log(f["N"]))
    mid = np.sqrt(f["N"][:-1] * f["N"][1:])
    ax.plot(mid, local, "o-", color=c, label=f"{label} (global {f['alpha']:.2f})")
    ax.axhline(f["alpha"], ls="--", color=c, lw=1, alpha=0.6)
ax.set(xscale="log", xlabel="N (geometric midpoint)", ylabel="local exponent",
       title="(d) local $d\\log T/d\\log N$: flat = true power law")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Section 10 - How sharply does sparsity matter?

June's minimum-density cell (`E = N-1`, k=2.0) versus this study's `E = 1.1N` (k=2.2). Both are
constant-mean-degree families differing by a 10% edge surplus, so the comparison isolates how
sensitive the Moran process is right at the spanning-tree boundary.

This is **not** a validation of the pipeline: the two are genuinely different graph families and
are expected to differ. It is a measurement in its own right, and a warning. If a 10% edge change
moves the answer a lot, the exponent fitted here does not transfer to the respiratory graphs
unless their mean degree also matches.

Both series are **fixation steps**, because that is the only one of the three that June's
`graph_statistics.csv` stores. June's raw shards would be needed for absorption steps or
simulation time, and are not read here.

In [ ]:
june = pd.read_csv(JUNE_BATCH / "graph_statistics.csv")
june = june[(june.category == "Random") & (june.n_edges == june.n_nodes - 1)]
june_curve = june.groupby("n_nodes")["mean_steps"].mean()  # CONDITIONAL = fixation steps
june_fit = fit_line(np.log(june_curve.index.values.astype(float)), np.log(june_curve.values))

print("Fixation steps, two constant-mean-degree families:")
print(f"  this study (k=2.2): alpha = {fit_fixation['alpha']:.3f}")
print(f"  June       (k=2.0): alpha = {june_fit['slope']:.3f}")
print(f"  difference        : {abs(fit_fixation['alpha'] - june_fit['slope']):.3f}")

shared = [n for n in curve_fixation.index if n in june_curve.index]
if shared:
    print("\nRatio at shared sizes (k=2.0 / k=2.2), i.e. the price of near-tree sparsity:")
    for n in shared:
        print(f"  N={n:>5}: June {june_curve[n]:>10,.0f}  this {curve_fixation.loc[n, 'T']:>10,.0f}"
              f"  ratio {june_curve[n] / curve_fixation.loc[n, 'T']:>6.1f}x")

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(june_curve.index, june_curve.values, "s--", color="gray",
        label=f"June, E=N-1, k=2.0 ($\\alpha$={june_fit['slope']:.2f})")
ax.plot(curve_fixation.index, curve_fixation["T"], "o-", color="darkorange",
        label=f"this study, E=1.1N, k=2.2 ($\\alpha$={fit_fixation['alpha']:.2f})")
ax.set(xscale="log", yscale="log", xlabel="N",
       ylabel="fixation steps", title="Sensitivity to sparsity at the tree boundary")
ax.legend()
plt.tight_layout()
plt.show()

## Section 11 - Throughput: is a step really O(1)?

`step()` is O(1) in work but not in *memory*: `state_`, `order_` and `loc_` all grow with N and
are accessed randomly, so throughput could fall at large N.

**The batch already answers this**, because it timed every run. `absorption_steps / sim_seconds`
per rung is a measured throughput curve over 26M runs on the real cluster, which is a far better
estimate than anything a single notebook cell can produce. Section 7 printed it, and it *rises*
with N: at k=2.2 the CSR array stays tiny (1100 edges at N=1000) so cache is never the binding
constraint, while the fixed per-run setup is amortised over ~92 steps at N=10 but ~111,000 at
N=1000.

The cell below is therefore a **portability check**, not the primary measurement: it re-measures
throughput on whatever machine you are sitting on and compares it to the cluster's.

**Run it in an `inode` session, not on the login node.**

In [ ]:
RUN_THROUGHPUT_CALIBRATION = False  # flip on inside an inode session

# The batch's own measured throughput, per rung. This is the reference the local
# calibration is compared against; it is what Section 12 costs from.
cluster_rate = curve_absorption["T"] / curve_seconds["T"]
STEPS_PER_SEC = float(cluster_rate.iloc[-1])  # large-N rate; cost lives at large N
print(f"Cluster throughput, measured over the batch itself: "
      f"{cluster_rate.iloc[0] / 1e6:.0f} M steps/s at N={cluster_rate.index[0]} "
      f"-> {STEPS_PER_SEC / 1e6:.0f} M/s at N={cluster_rate.index[-1]} (RISES with N)")

if not RUN_THROUGHPUT_CALIBRATION:
    print("\nLocal calibration skipped. Set RUN_THROUGHPUT_CALIBRATION = True in an "
          "inode session\nto compare this machine against the cluster.")
else:
    rows = []
    for n in LADDER:
        g = PopulationGraph.random_connected_graph(
            n_nodes=n, n_edges=n_edges_for(n), seed=0
        )
        # Size the repeat count from the MEASURED absorption steps at this rung, so
        # every point does roughly the same amount of work.
        reps = max(20, int(2e7 / max(curve_absorption.loc[n, "T"], 1)))
        sim = CppMoranProcess(
            graph_core=g.to_simulation_struct(),
            selection_coefficient=R_VALUES[0],
            max_steps=MAX_STEPS,
            seed=0,
        )
        t0 = time.perf_counter()
        out = sim.run_repeats(reps)
        wall = time.perf_counter() - t0
        steps = int(np.sum(out["steps"]))
        rows.append(dict(N=n, reps=reps, steps=steps, wall_sec=wall,
                         steps_per_sec=steps / wall))
        print(f"  N={n:>5}  {reps:>7} reps  {steps / wall / 1e6:>7.1f} M steps/s "
              f"(cluster: {cluster_rate.loc[n] / 1e6:>6.1f})")

    thru = pd.DataFrame(rows).set_index("N")
    ratio = float(thru.steps_per_sec.iloc[-1]) / STEPS_PER_SEC
    print(f"\nThis machine is {ratio:.2f}x the cluster at N={LADDER[-1]}.")
    STEPS_PER_SEC_MEASURED = float(thru.steps_per_sec.iloc[-1])

## Section 12 - How long do I actually wait?

This is the question the notebook is named after, and `sim_seconds` alone does not answer it.
`duration` times the C++ absorption loop and nothing else, so the wait is built in three layers,
each measured here rather than assumed:

| layer | what it adds | measured from |
|---|---|---|
| **1. simulation time** | the absorption loop | `duration` in the raw shards |
| **2. process CPU time** | mutant placement, Python, parquet writes | LSF `CPU time` in the job logs |
| **3. job wall time** | interpreter startup, shard load, I/O stalls | LSF `Run time` in the job logs |

Layers 2 and 3 are pure overhead in the sense that no simulation happens in them, but you wait
for them all the same, so a wall-clock promise built on layer 1 alone is a promise you will miss.

And there is a fourth thing that is not a factor at all: **the wait is the slowest job, not the
mean job.** `_create_task_list` balances workers by simulation *count*, so a worker that draws
N=1000 work does ~1000x the work of one that draws N=10, and you wait for that one. This is not a
small correction. The Section 2 pilot predicted a worst worker of ~0.4 min by dividing total work
by job count; the array's slowest job actually ran for 21.7 min, a 54x miss, and every bit of it
is imbalance rather than a bad throughput estimate.

And the imbalance is **topological, not just a size effect**. At a fixed N=1000 the four
topologies in this zoo span a factor of ~670 in seconds per run, because absorption time is a
property of the graph and not of its node count. The slowest job in this batch spent 91% of its
21 minutes on `star_n1000` alone. So the last cell breaks the wait down per topology as well as
per rung: a promise made from the random graphs alone understates the worst case by ~91x.

Queue time is excluded throughout. It is real, it is often the largest term, and it is not a
property of the simulation.

In [ ]:
import re

# --- layer 1: simulation time, summed over every run in the batch -----------
job_speed = pd.read_csv(BATCH_DIR / "job_speed.csv")
in_loop_h = job_speed.duration.sum() / 3600
loop_rate = job_speed.steps.sum() / job_speed.duration.sum()

# --- layers 2 and 3: what LSF actually charged, and how long it actually took
# Read from the array's own logs rather than modelled. Absent on a cleaned batch,
# in which case both factors fall back to 1.0 and the cell says so.
array_id = INFO["hpc"]["lsf_job_id"]
logs = sorted((BATCH_DIR / "logs").glob(f"job_{array_id}_*.out"))
cpu_s, wall_s = [], []
for f in logs:
    text = f.read_text(errors="ignore")
    m_cpu = re.search(r"CPU time :\s+([\d.]+) sec", text)
    m_run = re.search(r"Run time :\s+(\d+) sec", text)
    if m_cpu:
        cpu_s.append(float(m_cpu.group(1)))
    if m_run:
        wall_s.append(float(m_run.group(1)))

have_logs = bool(wall_s)
cpu_h = sum(cpu_s) / 3600 if cpu_s else in_loop_h
wall_h = sum(wall_s) / 3600 if wall_s else in_loop_h
CPU_OVERHEAD = cpu_h / in_loop_h
WALL_OVERHEAD = wall_h / in_loop_h

print(f"{len(logs)} job logs read" if have_logs
      else "NO job logs found - overhead factors fall back to 1.0x")
print(f"  layer 1  simulation time : {in_loop_h:6.3f} core-h  "
      f"({loop_rate / 1e6:.1f} M steps/s)")
print(f"  layer 2  process CPU time: {cpu_h:6.3f} core-h  {CPU_OVERHEAD:.2f}x layer 1")
print(f"  layer 3  job wall time   : {wall_h:6.3f} core-h  {WALL_OVERHEAD:.2f}x layer 1")

if have_logs:
    w = np.array(wall_s)
    print(f"\n  per job wall: median {np.median(w):.0f} s | p90 {np.percentile(w, 90):.0f} s "
          f"| max {w.max():.0f} s = {w.max() / 60:.1f} min")
    print(f"  You wait for the MAX, not the mean ({w.mean():.0f} s): the array is only "
          f"done when its\n  unluckiest worker is, and that worker drew the N={LADDER[-1]} work.")

# --- the headline: one graph at the top of the ladder ----------------------
N_BIG = LADDER[-1]
sec_run = curve_seconds.loc[N_BIG, "T"]           # mean over ALL runs
rho_big = raw.loc[raw.n_nodes == N_BIG, "rho"].mean()
sec_fix = curve_fixation.loc[N_BIG, "T"] / float(cluster_rate.loc[N_BIG])  # fixating only

bar = "=" * 70
print(f"\n{bar}")
print(f"HOW LONG TO RUN ONE N={N_BIG} GRAPH   (k={K_MEAN}, r={R_VALUES[0]}, one core)")
print(bar)
print(f"  one run, averaged over all outcomes : {sec_run * 1e3:9.3f} ms")
print(f"  one run that actually fixates       : {sec_fix * 1e3:9.3f} ms   "
      f"({curve_fixation.loc[N_BIG, 'T'] / curve_absorption.loc[N_BIG, 'T']:.1f}x longer, "
      f"and only {rho_big:.1%} of runs)")
print(f"  {N_REPEATS:,} runs, one full cell        : {sec_run * N_REPEATS:9.1f} s   "
      f"-> {sec_run * N_REPEATS * WALL_OVERHEAD:.1f} s of real waiting "
      f"({WALL_OVERHEAD:.2f}x)")
print(f"  one rung of {GRAPHS_PER_RUNG} graphs           : "
      f"{sec_run * N_REPEATS * GRAPHS_PER_RUNG / 3600:9.2f} core-h "
      f"-> {sec_run * N_REPEATS * GRAPHS_PER_RUNG * WALL_OVERHEAD / 3600:.2f} charged")
for n_cores in (10, 100, 1000):
    par = sec_run * N_REPEATS * GRAPHS_PER_RUNG * WALL_OVERHEAD / n_cores / 60
    print(f"      spread over {n_cores:>4} cores      : {par:9.1f} min wall clock")
print("      (perfect balance assumed; real arrays are limited by the slowest worker)")

# --- the same question, per topology, at the top of the ladder --------------
# The random k=2.2 graphs are what this study is ABOUT, but they are not what a
# batch costs: the baselines share the ladder and absorption time is a property of
# the topology. Reporting only the Random row is how a 20-minute job comes as a
# surprise.
top = raw[raw.n_nodes == N_BIG].groupby("category").agg(
    ms_per_run=("sim_seconds", lambda s: s.mean() * 1e3),
    absorption_steps=("absorption_steps", "mean"),
    rho=("rho", "mean"),
)
for reps in (N_REPEATS, 10 * N_REPEATS):
    top[f"{reps // 1000}k_min"] = top.ms_per_run / 1e3 * reps / 60
top = top.sort_values("ms_per_run")

print(f"\nOne graph at N={N_BIG}, one core, by topology "
      f"(minutes include no wall overhead):")
print(top.to_string(formatters={
    "ms_per_run": lambda v: f"{v:,.2f}",
    "absorption_steps": lambda v: f"{v:,.0f}",
    "rho": lambda v: f"{v:.3f}",
    **{c: (lambda v: f"{v:,.2f}") for c in top.columns if c.endswith("_min")},
}))
slow, fast = top.ms_per_run.iloc[-1], top.ms_per_run.iloc[0]
print(f"  spread at fixed N={N_BIG}: {slow / fast:,.0f}x between "
      f"{top.index[-1]} and {top.index[0]}, and "
      f"{slow / top.loc['Random', 'ms_per_run']:.0f}x above the Random graphs this "
      f"study is about.")

# Cross-check the per-run numbers against the batch total, which was measured a
# completely different way (per-job sums in job_speed.csv).
predicted_h = (curve_seconds["T"] * N_REPEATS * curve_seconds["n_graphs"]).sum() / 3600
print(f"\nCross-check: per-run means predict {predicted_h:.2f} core-h for the random "
      f"graphs alone;\njob_speed.csv reports {in_loop_h:.2f} core-h for every graph, "
      f"including the dense Complete baselines.")

## Section 13 - What would a bigger experiment cost?

The payoff, and it is quoted in **seconds**, extrapolated from the measured `sim_seconds` fit
rather than from steps divided by an assumed throughput.

Two extrapolations are shown, because Section 9's curvature test decides which one is honest:

- **global**: the single `alpha` fitted across the whole ladder. Correct only if the local
  exponent is flat.
- **local, anchored**: `alpha` fitted on the top half only, anchored at the *measured* N=1000
  point. This is the right one when the exponent drifts, because it uses the behaviour at the end
  of the ladder you are actually extending from, and it passes exactly through the last real
  measurement instead of through a global average.

They diverge by about a factor of 2 at N=10000. The 95% band from the global `alpha` CI is drawn
as well, and it is worth noting that the band is *wider* than the gap between the two models: the
sampling error on a single fitted exponent already spans more than the disagreement about which
exponent to use. Neither is small. Quote a range out there, never a number.

In [ ]:
f = fit_seconds  # THE WAIT. Costing from fixation_steps would be a category error.
rate_note = ("this machine" if "STEPS_PER_SEC_MEASURED" in globals() else "the cluster")

TARGETS = [1000, 1500, 2000, 3000, 5000, 10000]
N_BIG = LADDER[-1]
sec_anchor = curve_seconds.loc[N_BIG, "T"]


def sec_global(n, a):
    """Seconds per run from the global power-law fit."""
    return np.exp(f["power"]["intercept"]) * np.asarray(n, float) ** a


def sec_local(n):
    """Seconds per run from the top-half exponent, anchored at the measured N_BIG."""
    return sec_anchor * (np.asarray(n, float) / N_BIG) ** f["alpha_hi"]


def core_h(sec_per_run):
    return sec_per_run * N_REPEATS * GRAPHS_PER_RUNG * WALL_OVERHEAD / 3600


print(f"Fitted on simulation time measured on {rate_note}.")
print(f"  global exponent     alpha = {f['alpha']:.3f}  (95% CI {f['lo']:.3f} .. {f['hi']:.3f})")
print(f"  top-half exponent   alpha = {f['alpha_hi']:.3f}  (N >= {f['split']:.0f}), "
      f"anchored at the measured {sec_anchor * 1e3:.3f} ms at N={N_BIG}")
print(f"  wall-clock overhead {WALL_OVERHEAD:.2f}x applied to every core-hour below")
print("  -> " + ("exponent is stable; the two agree"
                 if f["stable"] else
                 "EXPONENT DRIFTS; trust the anchored local model, not the global one"))

extrap = pd.DataFrame({"N": TARGETS})
extrap["ms_global"] = sec_global(TARGETS, f["alpha"]) * 1e3
extrap["ms_local"] = sec_local(TARGETS) * 1e3
extrap["core_h_global"] = core_h(sec_global(TARGETS, f["alpha"]))
extrap["core_h_local"] = core_h(sec_local(TARGETS))
extrap["core_h_lo"] = core_h(sec_global(TARGETS, f["lo"]))
extrap["core_h_hi"] = core_h(sec_global(TARGETS, f["hi"]))
extrap["min_1000_cores"] = extrap.core_h_local * 60 / 1000

print(f"\nOne rung of {GRAPHS_PER_RUNG} graphs at {N_REPEATS:,} repeats, r={R_VALUES}:")
print(f"{'N':>7} {'ms per run':>19} {'core-hours per rung':>21} {'min on':>9}")
print(f"{'':>7} {'global':>9} {'local':>9} {'global':>10} {'local':>10} "
      f"{'1000 cores':>10}")
for row in extrap.itertuples():
    print(f"{row.N:>7} {row.ms_global:>9.2f} {row.ms_local:>9.2f} "
          f"{row.core_h_global:>10.2f} {row.core_h_local:>10.2f} {row.min_1000_cores:>10.1f}")

worst = extrap.iloc[-1]
print(f"\nAt N={worst.N:.0f}: the two models differ by "
      f"{worst.core_h_global / worst.core_h_local:.1f}x, and the global 95% band alone spans "
      f"{worst.core_h_hi / worst.core_h_lo:.1f}x.\nBoth are large. Quote a range out there, "
      f"never a number.")
print(f"MAX_STEPS headroom: the largest predicted absorption is "
      f"{np.exp(fit_absorption['power']['intercept']) * TARGETS[-1] ** fit_absorption['alpha']:,.0f} "
      f"steps against a {MAX_STEPS:,} cap.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
ax.errorbar(curve_seconds.index, curve_seconds["T"] * 1e3,
            yerr=curve_seconds["sem"] * 1e3, fmt="o", color="seagreen", label="measured")
grid = np.logspace(1, np.log10(TARGETS[-1]), 100)
ax.plot(grid, sec_global(grid, f["alpha"]) * 1e3, "-", color="crimson",
        label=f"global $N^{{{f['alpha']:.2f}}}$")
ax.plot(grid, sec_local(grid) * 1e3, "--", color="navy",
        label=f"local $N^{{{f['alpha_hi']:.2f}}}$, anchored")
ax.axvline(N_BIG, color="gray", ls=":", lw=1)
ax.set(xscale="log", yscale="log", xlabel="N (nodes)", ylabel="ms per run (1 core)",
       title="Simulation time per run, and two extrapolations")
ax.legend(fontsize=8)

ax = axes[1]
ax.fill_between(extrap.N, extrap.core_h_lo, extrap.core_h_hi, alpha=0.2,
                color="crimson", label="95% band, global $\\alpha$")
ax.plot(extrap.N, extrap.core_h_global, "o-", color="crimson", label="global")
ax.plot(extrap.N, extrap.core_h_local, "s--", color="navy", label="local, anchored")
ax.axhline(in_loop_h, ls=":", color="gray", label=f"this whole study ({in_loop_h:.1f} core-h)")
ax.set(xscale="log", yscale="log", xlabel="N (nodes)",
       ylabel=f"core-hours per rung of {GRAPHS_PER_RUNG}",
       title=f"Cost to extend the ladder (incl. {WALL_OVERHEAD:.2f}x wall overhead)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Section 14 - Conclusion

Fill the table from Section 9's output. There are **three** answers, not one, and they are not
interchangeable:

| quantity | alpha | 95% CI | delta AIC vs exponential | alpha stable across the ladder? |
|---|---|---|---|---|
| fixation steps (biology) | ____ | ____ | ____ | ____ |
| absorption steps (work) | ____ | ____ | ____ | ____ |
| simulation time (the wait) | ____ | ____ | ____ | ____ |

- **Linear ruled out?** ____
- **Exponential ruled out?** ____
- **The wait for one N=1000 graph at 10,000 repeats:** ____ s on one core (Section 12).
- **Practical ceiling:** at ____ core-hours per rung, N = ____ is the largest size worth running
  at these repeat counts.

### Caveats to carry forward

1. **The exponent drifts, so there is no single law.** Section 9's curvature test is the thing to
   read before quoting any number. Where it drifts, `T ~ N^alpha` is a local description, not a
   law, and Section 13's *anchored local* model is the honest extrapolation. Extending the ladder
   past 1000 is the fix; at this cost it is nearly free.
2. **Three quantities, three exponents.** Fixation steps, absorption steps and simulation time do
   not share one. Quoting a single number as "the" scaling of this simulation is wrong, and which
   one you want depends on whether you are describing evolution, counting work, or planning a wait.
3. **Simulation time is not the wait.** `duration` covers the C++ absorption loop only. Section 12
   measures the CPU and wall-clock factors on top of it from the job logs, and they are not 1.0.
   The wait is also set by the slowest worker in the array, never the mean, and queue time is
   excluded from all of it.
4. **One selection coefficient.** Everything is r=1.1. Absorption behaves very differently near
   r=1 (diffusive rather than driven), so any cost estimate for an r-sweep including near-neutral
   values is understated.
5. **One sparsity, and sparsity matters sharply.** k=2.2 only. Section 10 quantifies the gap to
   k=2.0. Do not reuse these exponents at another density, and note that the respiratory graphs
   sit in exactly this sensitive near-tree region.
6. **The batch is 1.6% incomplete** (Section 6: 16 of 1000 shards missing, 32 short cells). Section
   7 quantifies the effect on every curve and it is under a percent, but the shortfall is real and
   it is why `verification.json` says FAIL.
7. **The Section 2 pilot is historical.** It was measured on a login node on 2026-08-23 at 5
   graphs x 400 repeats per rung, and it only ever served to budget the launch. Every number it
   fed is recomputed from the real batch in Sections 7 to 13.